# Alumni Mentorship Matching Engine — Notebook Edition

This notebook is a single-file, self-contained copy of the **retrieve-then-rank student–alumni matching engine** (the `matching_engine/` package plus `run_demo.py`). Every module's logic is reproduced here in order, each preceded by an explanation of what the cell does and why.

**Structure**

| Part | Contents |
|---|---|
| 1. Setup | Imports and configuration |
| 2. Engine | `constants` → `data` → `sampling` → `synthetic_profile` → `features` → `retrieval` → `ranking` → `assignment` → `evaluation` → `events` → `feedback_schema` → `export` |
| 3. End-to-end demo | The `run_demo.py` pipeline, split into one cell per step |

**Architecture in one paragraph.** Each person is turned into one weighted, sparse feature vector (major, hobbies, personal traits/career goals, location, career track). Day-to-day matching is a *live, per-student query*: retrieve the nearest mentors from a pre-built index, drop mentors with no remaining capacity, then diversify the shortlist with Maximal Marginal Relevance (MMR). For cold-start batches (e.g. the initial pilot cohort), a Hungarian-algorithm assignment maximizes total compatibility subject to each mentor's capacity — and doubles as the "ceiling" the live flow is measured against.

> ⚠️ **Synthetic data caveat.** No real opt-in or feedback forms exist yet. The opt-in population is a random sample, and the opt-in-form fields (career goals, capacity, etc.) are synthesized from existing profile fields. This is illustrative data for exercising the pipeline — never real user input, and never used for training.

## 1. Setup

### 1.1 Dependencies
Runtime requirements (same as `requirements.txt`): `pandas>=2.0`, `numpy>=1.24`, `scipy>=1.10`, `scikit-learn>=1.3`. Uncomment the line below to install them into the notebook's kernel if needed.

In [ ]:
# %pip install "pandas>=2.0" "numpy>=1.24" "scipy>=1.10" "scikit-learn>=1.3"

### 1.2 Imports
All imports used anywhere in the engine, consolidated in one place (in the package they were spread across the individual modules):

- **Standard library** — `json` for loading data / writing exports and event logs; `dataclasses` for the `Weights`, `MatchEvent` and `FeedbackForm` record types; `datetime` for event/feedback timestamps; `typing` for type hints.
- **numpy / pandas** — numeric arrays and tabular profile data.
- **scipy** — `sparse` matrices (feature vectors stay sparse end-to-end) and `linear_sum_assignment` (the Hungarian algorithm).
- **scikit-learn** — TF-IDF text vectorization, one-hot encoding, L2 normalization, cosine similarity, nearest-neighbour search and the train/test split.

In [ ]:
import json
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from typing import Any, Literal, Optional

import numpy as np
import pandas as pd
from scipy import sparse
from scipy.optimize import linear_sum_assignment
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import OneHotEncoder, normalize

### 1.3 Configuration
Run-level settings for the end-to-end demo (Part 3):

| Setting | Meaning |
|---|---|
| `DATA_PATH` | Full population file (23,236 profiles). Must sit next to this notebook. |
| `SEED` | Makes sampling, synthetic fields and the train/live split reproducible. |
| `ALUMNI_OPT_IN_N` / `STUDENT_OPT_IN_N` | Size of the simulated opted-in pilot population (~5% of each pool, preserving the ~1:3 alumni:student ratio). |
| `LIVE_FRAC` | Fraction of the opted-in population held out as "live" traffic. |
| `OUTPUT_PATH` / `EVENT_LOG_PATH` | Where the demo writes the human-review export and the example event log. |

In [ ]:
DATA_PATH = "student_profiles.json"
SEED = 42
ALUMNI_OPT_IN_N = 300
STUDENT_OPT_IN_N = 900
LIVE_FRAC = 0.3
OUTPUT_PATH = "match_results.json"
EVENT_LOG_PATH = "match_events.jsonl"

## 2. The matching engine

Each subsection below reproduces one module of the `matching_engine` package.

### 2.1 Shared vocabulary (`constants.py`)
Fixed vocabularies used by both feature encoding and synthetic-field generation, kept in one place so those two parts don't depend on each other:

- `TRACKS` — the 8 career tracks presented on the platform's landing page. Used as the fixed vocabulary for the multi-hot "track" feature block.
- `CADENCES`, `FORMATS`, `SUPPORT_OPTIONS` — allowed values for meeting cadence, meeting format and the kinds of support a student can request.

In [ ]:
# The 8 career tracks already presented on the platform's landing page.
TRACKS = [
    "Data Science",
    "Software Engineering",
    "Product Design",
    "Finance",
    "Entrepreneurship",
    "Public Policy",
    "Marketing",
    "Research",
]

CADENCES = ["weekly", "biweekly", "as-needed"]
FORMATS = ["virtual", "in-person", "either"]
SUPPORT_OPTIONS = ["resume_review", "interview_prep", "networking", "general_guidance"]

### 2.2 Loading the population (`data.py`)

- **`load_data(path)`** — loads a `.json` (list of records) or `.csv` file into a DataFrame. CSV files often store `Hobbies` as a comma-separated string, so it is normalized into a Python list. Every row is given a stable `person_id` (`P0`, `P1`, …) used as the key throughout the engine.
- **`split_alumni_students(df)`** — proxy split: `Year == "Senior"` → alumni-eligible pool, everyone else → student pool. This is a *proxy*, not real alumni status, and it is the full population before any opt-in filtering.

In [ ]:
def load_data(path: str) -> pd.DataFrame:
    """Loads either a .json (list of records) or .csv file into a DataFrame.
    Ensures 'Hobbies' is always a Python list (CSV often stores it as a
    comma/semicolon separated string)."""
    if path.endswith(".json"):
        with open(path) as f:
            records = json.load(f)
        df = pd.DataFrame(records)
    else:
        df = pd.read_csv(path)

    if df["Hobbies"].apply(lambda x: isinstance(x, str)).any():
        df["Hobbies"] = df["Hobbies"].apply(
            lambda x: [h.strip() for h in x.split(",")] if isinstance(x, str) else x
        )

    df = df.reset_index(drop=True)
    df["person_id"] = df.index.map(lambda i: f"P{i}")
    return df


def split_alumni_students(df: pd.DataFrame, alumni_years=("Senior",)):
    """Proxy split: Year == 'Senior' -> alumni-eligible pool, everyone else ->
    student pool. This is the full population, before any opt-in filtering --
    see matching_engine.sampling for turning this into the opted-in subset
    the mentorship program actually operates on."""
    alumni = df[df["Year"].isin(alumni_years)].reset_index(drop=True)
    students = df[~df["Year"].isin(alumni_years)].reset_index(drop=True)
    return alumni, students

### 2.3 Simulated opt-in and train/live split (`sampling.py`)

There is no real opt-in data yet, so a random sample of each pool stands in for "the people who opted in".

- **`sample_opt_in_population(...)`** — randomly samples `alumni_n` alumni and `student_n` students (capped at the pool size so a smaller dataset doesn't error).
- **`train_live_split(df, live_frac)`** — splits an opted-in pool into a **train** slice (used to fit the feature encoder) and a held-out **live** slice (used to demonstrate the online recommend/assignment flow as if it were production traffic — nobody in `live` is seen during fitting).

In [ ]:
def sample_opt_in_population(
    alumni_df: pd.DataFrame,
    students_df: pd.DataFrame,
    seed: int = 42,
    alumni_n: int = 300,
    student_n: int = 900,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Randomly samples `alumni_n` alumni and `student_n` students to stand in
    for the opted-in population.

    Defaults (~5% of each pool, preserving the observed ~1:3 alumni:student
    ratio) are a pilot-scale choice: large enough that capacity constraints,
    the assignment solver, and MMR diversification all have something
    meaningful to do, small enough to run fast and stay easy to eyeball.
    Tune via the arguments once you have a real target pilot size (e.g. a
    single school).

    `n` is capped at the pool size so this doesn't error out on a smaller
    dataset (e.g. the old 600-record mock).
    """
    alumni_n = min(alumni_n, len(alumni_df))
    student_n = min(student_n, len(students_df))

    opted_in_alumni = alumni_df.sample(n=alumni_n, random_state=seed).reset_index(drop=True)
    opted_in_students = students_df.sample(n=student_n, random_state=seed).reset_index(drop=True)
    return opted_in_alumni, opted_in_students


def train_live_split(
    df: pd.DataFrame, live_frac: float = 0.3, seed: int = 42
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Splits an opted-in pool into a train slice and a held-out live slice.

    `train`: used to fit/tune the feature encoder and (once real feedback
    exists) any learned component.
    `live`: held out to demonstrate the actual online recommend/assignment
    flow as if it were production traffic -- nobody in `live` was seen
    during fitting.

    Applied identically to the sampled alumni and the sampled students; the
    two calls are independent (an alumnus's train/live assignment has no
    bearing on which split a given student lands in).
    """
    train_df, live_df = train_test_split(df, test_size=live_frac, random_state=seed)
    return train_df.reset_index(drop=True), live_df.reset_index(drop=True)

### 2.4 Synthetic opt-in-form fields (`synthetic_profile.py`)

The raw data has no career-goal or mentor-join-form fields, so this cell generates **plausible** values derived from each person's existing profile (Major, Unique Quality) rather than pure noise. Deterministic for a given seed.

- **`MAJOR_TO_TRACK`** / **`map_major_to_track`** — best-fit mapping of the 70 observed majors onto the 8 tracks (unknown majors default to `"Research"`).
- **`_sample_subset`** — helper that picks 1–2 distinct options from a list (used for `requested_support`).
- **`_GOAL_TEMPLATES`** — sentence templates used to write a free-text `career_goal_text`.
- **`synthesize_student_fields(df)`** — adds `target_track` (major-derived, but with probability `pivot_prob` a random track to simulate students pivoting careers), `career_goal_text`, `preferred_cadence`, `format_preference`, `requested_support` and `consent`.
- **`synthesize_alumni_fields(df)`** — adds a per-mentor `capacity` (skewed toward small numbers: most mentors take 1–2 students), `mentor_tracks` (own major-derived track, plus a possible second track), `availability_cadence`, `format_preference`, `verified` and `consent`.

Both functions return a **copy**; the input DataFrame is not mutated.

In [ ]:
# Best-fit mapping of the 70 majors observed in student_profiles.json onto
# the 8 tracks. Necessarily approximate for majors with no clean match
# (humanities/arts/health-science majors mostly fall to "Research" as the
# broad academic/clinical catch-all) -- good enough as a synthesis seed,
# not meant to be authoritative career guidance.
MAJOR_TO_TRACK: dict[str, str] = {
    # Data Science
    "Mathematics": "Data Science",
    "Information Systems": "Data Science",
    # Software Engineering
    "Computer Science": "Software Engineering",
    "Computer Engineering": "Software Engineering",
    "Electrical Engineering": "Software Engineering",
    # Product Design
    "Architecture": "Product Design",
    "Fashion Design": "Product Design",
    "Graphic Communication": "Product Design",
    "Graphic Design": "Product Design",
    "Industrial Design": "Product Design",
    "Interior Design": "Product Design",
    # Finance
    "Accounting": "Finance",
    "Economics": "Finance",
    # Entrepreneurship
    "Business Administration": "Entrepreneurship",
    "Culinary Arts": "Entrepreneurship",
    "Hospitality Management": "Entrepreneurship",
    "Management": "Entrepreneurship",
    # Public Policy
    "Criminology": "Public Policy",
    "Early Childhood Education": "Public Policy",
    "Elementary Education": "Public Policy",
    "Environmental Science": "Public Policy",
    "International Relations": "Public Policy",
    "Political Science": "Public Policy",
    "Public Health": "Public Policy",
    "Social Work": "Public Policy",
    "Sociology": "Public Policy",
    # Marketing
    "Art Therapy": "Marketing",
    "Communication": "Marketing",
    "Dance": "Marketing",
    "Drama": "Marketing",
    "English Literature": "Marketing",
    "Film Studies": "Marketing",
    "Journalism": "Marketing",
    "Marketing": "Marketing",
    "Music": "Marketing",
    "Theater": "Marketing",
    # Research (broad STEM / humanities / clinical catch-all)
    "Aerospace Engineering": "Research",
    "Anthropology": "Research",
    "Archaeology": "Research",
    "Art History": "Research",
    "Biochemistry": "Research",
    "Biology": "Research",
    "Biomedical Engineering": "Research",
    "Botany": "Research",
    "Chemical Engineering": "Research",
    "Civil Engineering": "Research",
    "Classics": "Research",
    "Dentistry": "Research",
    "Earth Science": "Research",
    "Ecology": "Research",
    "Forensic Science": "Research",
    "Geography": "Research",
    "Geology": "Research",
    "Health Science": "Research",
    "History": "Research",
    "Kinesiology": "Research",
    "Linguistics": "Research",
    "Marine Biology": "Research",
    "Mechanical Engineering": "Research",
    "Meteorology": "Research",
    "Microbiology": "Research",
    "Molecular Biology": "Research",
    "Neuroscience": "Research",
    "Nuclear Engineering": "Research",
    "Nursing": "Research",
    "Nutrition Science": "Research",
    "Philosophy": "Research",
    "Physics": "Research",
    "Psychology": "Research",
    "Veterinary Science": "Research",
}


def map_major_to_track(major: str) -> str:
    """Looks up a major's best-fit track, defaulting to 'Research' for any
    major not in the table (keeps this forward-compatible with new majors
    appearing in future data without raising)."""
    return MAJOR_TO_TRACK.get(major, "Research")


def _sample_subset(rng: np.random.Generator, options: list[str], min_n=1, max_n=2) -> list[str]:
    size = int(rng.integers(min_n, max_n + 1))
    return sorted(rng.choice(options, size=size, replace=False).tolist())


_GOAL_TEMPLATES = [
    "Looking to grow into a {track} career, building on my {major} background and interest in {quality}.",
    "Exploring {track} as a career direction -- I'm studying {major} and drawn to {quality}.",
    "Hoping to find a mentor in {track} who can help me turn my {major} studies and interest in {quality} into a career.",
]


def synthesize_student_fields(df: pd.DataFrame, seed: int = 42, pivot_prob: float = 0.25) -> pd.DataFrame:
    """Adds the student career-goals-form fields to `df` (a copy; `df` itself
    is not mutated): `target_track`, `career_goal_text`, `preferred_cadence`,
    `format_preference`, `requested_support`, `consent`.

    `target_track` is mostly derived from `Major` via `MAJOR_TO_TRACK`, but
    with probability `pivot_prob` a different, uniformly-random track is
    used instead -- simulating the realistic minority of students who want
    to pivot away from their field of study rather than continue in it.
    """
    out = df.copy()
    rng = np.random.default_rng(seed)
    n = len(out)

    base_track = out["Major"].map(map_major_to_track)
    pivot_mask = rng.random(n) < pivot_prob
    random_track = rng.choice(TRACKS, size=n)
    out["target_track"] = np.where(pivot_mask, random_track, base_track)

    out["preferred_cadence"] = rng.choice(CADENCES, size=n)
    out["format_preference"] = rng.choice(FORMATS, size=n)
    out["requested_support"] = [_sample_subset(rng, SUPPORT_OPTIONS) for _ in range(n)]

    templates = rng.choice(_GOAL_TEMPLATES, size=n)
    out["career_goal_text"] = [
        templates[i].format(
            track=out["target_track"].iloc[i],
            major=out["Major"].iloc[i],
            quality=str(out["Unique Quality"].iloc[i]).lower(),
        )
        for i in range(n)
    ]

    out["consent"] = True
    return out


def synthesize_alumni_fields(
    df: pd.DataFrame,
    seed: int = 42,
    capacity_choices=(1, 2, 3, 4, 5),
    capacity_probs=(0.35, 0.30, 0.20, 0.10, 0.05),
    second_track_prob: float = 0.3,
) -> pd.DataFrame:
    """Adds the alumni mentor-join-form fields to `df` (a copy): `capacity`,
    `mentor_tracks`, `availability_cadence`, `format_preference`,
    `verified`, `consent`.

    `capacity` is deliberately NOT uniform -- it's drawn from a distribution
    weighted toward smaller numbers (most mentors take on a modest few,
    fewer take many), replacing the old pipeline's single hardcoded
    `capacity=9` for everyone. `mentor_tracks` is mostly the alumnus's own
    major-derived track, with a chance of a second track added (someone's
    current career often isn't identical to what they studied).
    """
    out = df.copy()
    rng = np.random.default_rng(seed)
    n = len(out)

    out["capacity"] = rng.choice(capacity_choices, size=n, p=capacity_probs)

    base_track = out["Major"].map(map_major_to_track)
    add_second = rng.random(n) < second_track_prob
    second_track = rng.choice(TRACKS, size=n)
    mentor_tracks = []
    for i in range(n):
        tracks = [base_track.iloc[i]]
        if add_second[i] and second_track[i] not in tracks:
            tracks.append(second_track[i])
        mentor_tracks.append(tracks)
    out["mentor_tracks"] = mentor_tracks

    out["availability_cadence"] = rng.choice(CADENCES, size=n)
    out["format_preference"] = rng.choice(FORMATS, size=n)
    # In reality this comes from the admin verification queue (already
    # built in the frontend); every synthetic mentor is treated as verified
    # since this is a test population, not a real moderation queue.
    out["verified"] = True
    out["consent"] = True
    return out

### 2.5 Feature engineering (`features.py`)

Turns each person into **one weighted, sparse compatibility vector**, so that cosine similarity between two vectors approximates a weighted sum of per-field similarities. Everything stays `scipy.sparse` end-to-end (no wasteful `.toarray()`).

- **`_track_lists` / `_encode_tracks`** — normalizes `mentor_tracks` (alumni, list) or `target_track` (students, single value) into lists and multi-hot encodes them over the fixed 8-track vocabulary. This is the closest thing the data has to an explicit stated preference.
- **`Weights`** — per-block weights (`major`, `hobbies`, `trait`, `location`, `track`).
- **`FeatureEncoder`**
  - `fit(df)` learns the TF-IDF vocabularies (Major, Hobbies, Unique Quality + career-goal text) and the one-hot categories (Country, State/Province).
  - `transform(df)` encodes each block, L2-normalizes it, builds a **location** block weighted 40 % country / 60 % state, multiplies each block by `sqrt(weight)` (so the weight — not its square — shows up in the dot product), and horizontally stacks the blocks into one CSR matrix.
- **`compatibility_score(a, b)`** — cosine similarity between two people's vectors; symmetric.

*Upgrade path:* TF-IDF measures lexical overlap so it runs offline; swapping in a sentence-transformers model would give semantic matching without changing anything downstream.

In [ ]:
_TRACK_INDEX = {t: i for i, t in enumerate(TRACKS)}


def _track_lists(df: pd.DataFrame) -> pd.Series:
    """Normalizes either `mentor_tracks` (list column) or `target_track`
    (single-value column) into a uniform Series of lists, so both roles can
    be encoded the same way. Rows with neither column, or missing values,
    get an empty list (contributes nothing to the track block, not an
    error)."""
    if "mentor_tracks" in df.columns:
        return df["mentor_tracks"].apply(lambda v: v if isinstance(v, list) else ([v] if pd.notna(v) else []))
    if "target_track" in df.columns:
        return df["target_track"].apply(lambda v: [v] if pd.notna(v) else [])
    return pd.Series([[] for _ in range(len(df))], index=df.index)


def _encode_tracks(df: pd.DataFrame) -> sparse.csr_matrix:
    """Multi-hot encoding over the fixed 8-track vocabulary. Not an sklearn
    transformer since the vocabulary is fixed and known ahead of time --
    nothing needs to be fit."""
    rows, cols, data = [], [], []
    for i, tracks in enumerate(_track_lists(df)):
        for t in tracks:
            j = _TRACK_INDEX.get(t)
            if j is not None:
                rows.append(i)
                cols.append(j)
                data.append(1.0)
    return sparse.csr_matrix((data, (rows, cols)), shape=(len(df), len(TRACKS)))


@dataclass
class Weights:
    major: float = 1.0
    hobbies: float = 1.0
    trait: float = 1.0
    location: float = 1.0
    track: float = 1.0


class FeatureEncoder:
    """Encodes Major, Hobbies, Unique Quality (+ career_goal_text when
    present), Country/State, and track preference into one concatenated,
    per-field-weighted sparse vector, such that cosine similarity on the
    concatenated vector approximates a weighted sum of the individual
    field similarities.

    NOTE ON TEXT ENCODING: TF-IDF (lexical overlap) by default so this runs
    fully offline. Swap `_trait_text`/`major_vec`/`hobby_vec` for a
    sentence-transformers model (e.g. 'all-MiniLM-L6-v2') for semantic
    matching if you have internet access -- see IMPLEMENTATION_GUIDE.md.
    """

    def __init__(self, weights: Weights = Weights()):
        self.weights = weights
        self.major_vec = TfidfVectorizer()
        self.hobby_vec = TfidfVectorizer()
        self.trait_vec = TfidfVectorizer()
        self.country_enc = OneHotEncoder(handle_unknown="ignore")
        self.state_enc = OneHotEncoder(handle_unknown="ignore")

    @staticmethod
    def _trait_text(df: pd.DataFrame) -> pd.Series:
        base = df["Unique Quality"].astype(str)
        if "career_goal_text" in df.columns:
            return base + " " + df["career_goal_text"].fillna("").astype(str)
        return base

    @staticmethod
    def _hobby_text(df: pd.DataFrame) -> pd.Series:
        return df["Hobbies"].apply(lambda h: " ".join(h))

    def fit(self, df: pd.DataFrame) -> "FeatureEncoder":
        self.major_vec.fit(df["Major"])
        self.hobby_vec.fit(self._hobby_text(df))
        self.trait_vec.fit(self._trait_text(df))
        self.country_enc.fit(df[["Country"]])
        self.state_enc.fit(df[["State/Province"]])
        return self

    def transform(self, df: pd.DataFrame) -> sparse.csr_matrix:
        w = self.weights
        major = normalize(self.major_vec.transform(df["Major"]))
        hobby = normalize(self.hobby_vec.transform(self._hobby_text(df)))
        trait = normalize(self.trait_vec.transform(self._trait_text(df)))
        country = normalize(self.country_enc.transform(df[["Country"]]))
        state = normalize(self.state_enc.transform(df[["State/Province"]]))
        # location = country match + state match, weighted 40/60 towards state
        location = normalize(sparse.hstack([country * 0.4, state * 0.6]))
        track = normalize(_encode_tracks(df))

        blocks = [
            np.sqrt(w.major) * major,
            np.sqrt(w.hobbies) * hobby,
            np.sqrt(w.trait) * trait,
            np.sqrt(w.location) * location,
            np.sqrt(w.track) * track,
        ]
        return sparse.hstack(blocks).tocsr()


def compatibility_score(vec_a: sparse.spmatrix, vec_b: sparse.spmatrix) -> float:
    """Cosine similarity between two people's combined feature vectors
    (each a 1-row sparse matrix, as returned by `.transform()` or a single
    row slice). Symmetric: score(a, b) == score(b, a)."""
    return float(cosine_similarity(vec_a, vec_b)[0, 0])

### 2.6 Candidate retrieval (`retrieval.py`)

Given a student, which mentors are worth ranking?

- **`MentorIndex`** — a cosine `NearestNeighbors` index built **once** over the mentor pool and queried on demand. `query(vec, k)` returns up to `k` `(mentor_id, similarity)` pairs, best first (similarity = `1 − cosine distance`). This is what makes repeated live queries cheap. Rebuild it when the mentor pool changes meaningfully.
- **`build_candidate_lists(...)`** — a batch top-K pass over every student in a single `kneighbors` call. Returns each student's ranked candidates plus a reverse index (mentor → {student: score}). Not needed by the assignment solver at pilot scale, but useful for exploration or as a fallback if the population outgrows a dense cost matrix.

In [ ]:
class MentorIndex:
    """A `NearestNeighbors` index over a fixed pool of mentor vectors, built
    once and queried on demand. Rebuild it (construct a new `MentorIndex`)
    whenever the underlying mentor pool changes meaningfully (e.g.
    periodically, or when capacity/opt-in status changes) rather than on
    every request.
    """

    def __init__(self, mentor_ids: list[str], mentor_vecs: sparse.spmatrix):
        if len(mentor_ids) != mentor_vecs.shape[0]:
            raise ValueError("mentor_ids and mentor_vecs must have the same length")
        self.mentor_ids = list(mentor_ids)
        self._nn = NearestNeighbors(metric="cosine")
        self._nn.fit(mentor_vecs)

    def query(self, vec: sparse.spmatrix, k: int = 30) -> list[tuple[str, float]]:
        """Returns up to `k` (mentor_id, similarity) pairs for one query
        vector (a 1-row sparse matrix), best match first."""
        k = min(k, len(self.mentor_ids))
        if k == 0:
            return []
        dist, idx = self._nn.kneighbors(vec, n_neighbors=k)
        ranked = [(self.mentor_ids[j], 1 - dist[0, pos]) for pos, j in enumerate(idx[0])]
        ranked.sort(key=lambda t: -t[1])
        return ranked


def build_candidate_lists(
    student_ids: list[str],
    student_vecs: sparse.spmatrix,
    alumni_ids: list[str],
    alumni_vecs: sparse.spmatrix,
    top_k: int = 30,
) -> tuple[dict[str, list[tuple[str, float]]], dict[str, dict[str, float]]]:
    """For every student at once, finds the top_k nearest mentors by cosine
    similarity (one batched `kneighbors` call, not a loop of single
    queries).

    Returns:
      student_candidates: {student_id: [(mentor_id, score), ...]} ranked
        best -> worst
      alumni_candidates: {mentor_id: {student_id: score, ...}} reverse
        index, used for e.g. MMR's near-miss pool
    """
    top_k = min(top_k, len(alumni_ids))
    index = MentorIndex(alumni_ids, alumni_vecs)
    dist, idx = index._nn.kneighbors(student_vecs, n_neighbors=top_k)

    student_candidates: dict[str, list[tuple[str, float]]] = {}
    alumni_candidates: dict[str, dict[str, float]] = {aid: {} for aid in alumni_ids}
    for i, sid in enumerate(student_ids):
        ranked = [(alumni_ids[j], 1 - dist[i, pos]) for pos, j in enumerate(idx[i])]
        ranked.sort(key=lambda t: -t[1])
        student_candidates[sid] = ranked
        for aid, score in ranked:
            alumni_candidates[aid][sid] = score

    return student_candidates, alumni_candidates

### 2.7 Ranking & live recommendation (`ranking.py`)

Turns retrieved candidates into what is actually shown to a student.

- **`mmr_select(...)`** — greedy **Maximal Marginal Relevance**. It picks the most relevant candidate first, then repeatedly picks the candidate maximizing `lam × relevance − (1 − lam) × max_similarity_to_already_selected`. `lam → 1` favours compatibility, `lam → 0` favours variety; `lam = 1` is plain top-k. This keeps a student from seeing five near-duplicate mentors.
- **`_matches_filters(row, filters)`** — a row passes if every filter value equals the row's value, or (for list columns like `mentor_tracks`) is contained in it.
- **`recommend_for_student(...)`** — **the live, single-student entry point**:
  1. With `hard_filters` → narrow the mentor pool by exact match (requires `mentor_df_indexed`), drop full mentors, and rank by direct cosine similarity.
  2. Without filters → query the shared `MentorIndex`, then drop mentors with no remaining capacity.
  3. MMR-diversify the surviving candidates down to `show_k` mentor ids.

In [ ]:
def mmr_select(candidate_ids, candidate_vecs, relevance_scores, k, lam=0.6):
    """Greedy MMR: iteratively picks the candidate that maximizes
        lam * relevance - (1 - lam) * max_similarity_to_already_selected
    lam close to 1.0 -> prioritize compatibility; lam close to 0.0 ->
    prioritize spreading the options out. lam=1.0 reduces to plain top-k by
    relevance. `candidate_vecs` may be sparse or dense -- `cosine_similarity`
    handles either, and the output pairwise matrix is small regardless
    (candidate pools here are at most a few dozen).
    """
    n = len(candidate_ids)
    k = min(k, n)
    if k == 0:
        return []
    sim = cosine_similarity(candidate_vecs)
    selected, remaining = [], list(range(n))

    first = int(np.argmax(relevance_scores))
    selected.append(first)
    remaining.remove(first)

    while len(selected) < k and remaining:
        best_i, best_score = None, -np.inf
        for i in remaining:
            max_sim_to_selected = max(sim[i, j] for j in selected)
            mmr = lam * relevance_scores[i] - (1 - lam) * max_sim_to_selected
            if mmr > best_score:
                best_score, best_i = mmr, i
        selected.append(best_i)
        remaining.remove(best_i)

    return [candidate_ids[i] for i in selected]


def _matches_filters(row: pd.Series, filters: dict[str, Any]) -> bool:
    """A row matches if, for every filter key, either the row's value for
    that column equals the wanted value, or (for list-valued columns like
    `mentor_tracks`) the wanted value is a member of the list."""
    for key, want in filters.items():
        have = row.get(key)
        if isinstance(have, list):
            if want not in have:
                return False
        elif have != want:
            return False
    return True


def recommend_for_student(
    student_vec: sparse.spmatrix,
    mentor_index: MentorIndex,
    id_to_vec: dict[str, sparse.spmatrix],
    remaining_capacity: dict[str, int],
    mentor_df_indexed: pd.DataFrame | None = None,
    hard_filters: dict[str, Any] | None = None,
    top_k_candidates: int = 30,
    show_k: int = 5,
    lam: float = 0.6,
) -> list[str]:
    """The live, single-student entry point: returns up to `show_k`
    recommended mentor ids, ranked and diversified.

    - If `hard_filters` is given (e.g. {'target_track': 'Data Science'},
      matched against a mentor's `mentor_tracks` list; or
      {'format_preference': 'virtual'}), the eligible pool is narrowed by
      exact match BEFORE ranking -- `mentor_df_indexed` (a mentor DataFrame
      indexed by `person_id`) is required in this case, since filters
      apply to raw columns the vector-only `mentor_index` doesn't carry.
      At pilot scale a filtered pool is small enough that a direct
      similarity computation is simpler than building a throwaway index.
    - With no `hard_filters`, `mentor_index` (built once, shared across
      many queries) is used directly -- this is the case where a
      pre-built index actually pays for itself.

    Either way, mentors with no remaining capacity are dropped before
    ranking, and the final list is MMR-diversified down to `show_k` so a
    student doesn't just see 5 near-duplicate options.
    """
    if hard_filters:
        if mentor_df_indexed is None:
            raise ValueError("hard_filters given but mentor_df_indexed is None")
        eligible = [pid for pid, row in mentor_df_indexed.iterrows() if _matches_filters(row, hard_filters)]
        eligible = [pid for pid in eligible if remaining_capacity.get(pid, 0) > 0]
        if not eligible:
            return []
        eligible_vecs = sparse.vstack([id_to_vec[pid] for pid in eligible])
        sims = cosine_similarity(student_vec, eligible_vecs)[0]
        candidates = sorted(zip(eligible, sims), key=lambda t: -t[1])[:top_k_candidates]
    else:
        raw = mentor_index.query(student_vec, k=top_k_candidates)
        candidates = [(pid, score) for pid, score in raw if remaining_capacity.get(pid, 0) > 0]

    if not candidates:
        return []

    cand_ids = [pid for pid, _ in candidates]
    cand_scores = np.array([s for _, s in candidates])
    cand_vecs = sparse.vstack([id_to_vec[pid] for pid in cand_ids])
    return mmr_select(cand_ids, cand_vecs, cand_scores, k=show_k, lam=lam)

### 2.8 Batch capacitated assignment (`assignment.py`)

For **cold-start sweeps only** (e.g. the initial pilot cohort, or a periodic pass over still-unmatched students) — not the day-to-day mechanism.

**`capacitated_assignment(...)`** solves one optimization: *maximize total compatibility subject to each mentor's own capacity*.

1. Cheaply sums the total seat count first, and raises a `ValueError` if the dense `students × seats` cost matrix would exceed `MAX_DENSE_CELLS` (4 M cells) — sample the population first rather than calling this at full scale.
2. Duplicates each mentor into `capacity` "seats".
3. Builds the cost matrix as **negative** cosine similarity (minimizing cost = maximizing similarity).
4. Solves it with the Hungarian algorithm (`linear_sum_assignment`).
5. Returns `(groups, total_score)` where `groups` maps mentor id → list of assigned student ids.

In [ ]:
# Safety guard: a dense n_students x n_seats cost matrix at this many cells
# is still comfortably fast and memory-light (a few tens of MB); well past
# it, this stops being the right tool -- sample the population first
# (matching_engine.sampling) rather than calling this on a full-scale one.
MAX_DENSE_CELLS = 4_000_000


def capacitated_assignment(
    student_ids: list[str],
    student_vecs,
    alumni_ids: list[str],
    alumni_vecs,
    alumni_capacity: dict[str, int],
) -> tuple[dict[str, list[str]], float]:
    """Optimal one-shot assignment maximizing total compatibility, subject
    to each mentor's own capacity. Returns (groups, total_score).

    Raises ValueError instead of silently hanging/exhausting memory if the
    resulting dense cost matrix would be too large -- see MAX_DENSE_CELLS.
    """
    n_students = len(student_ids)
    # Check the seat count cheaply (an O(n_alumni) sum) before building the
    # actual seats list -- a pathological capacity value (a typo'd extra
    # zero or two) would otherwise let the list comprehension itself try to
    # allocate tens of millions of tuples before the guard ever runs.
    total_seats = sum(int(alumni_capacity.get(aid, 0)) for aid in alumni_ids)

    if n_students == 0 or total_seats == 0:
        return {aid: [] for aid in alumni_ids}, 0.0

    cell_count = n_students * total_seats
    if cell_count > MAX_DENSE_CELLS:
        raise ValueError(
            f"capacitated_assignment would build a {n_students}x{total_seats} "
            f"({cell_count:,}-cell) dense cost matrix, over the "
            f"{MAX_DENSE_CELLS:,}-cell safety limit. This only scales to a "
            "few thousand people per side -- sample the population first "
            "(matching_engine.sampling.sample_opt_in_population) rather "
            "than calling this directly on a full-scale population."
        )

    seats = [(aid, seat) for aid in alumni_ids for seat in range(int(alumni_capacity.get(aid, 0)))]
    alumni_idx = {aid: i for i, aid in enumerate(alumni_ids)}
    sim_full = cosine_similarity(student_vecs, alumni_vecs)  # small pool at this scale, dense is fine
    cost = np.empty((n_students, len(seats)))
    for j, (aid, _seat) in enumerate(seats):
        cost[:, j] = -sim_full[:, alumni_idx[aid]]  # negate: minimize == maximize

    row_ind, col_ind = linear_sum_assignment(cost)
    groups: dict[str, list[str]] = {aid: [] for aid in alumni_ids}
    total_score = 0.0
    for r, c in zip(row_ind, col_ind):
        aid, _seat = seats[c]
        groups[aid].append(student_ids[r])
        total_score += -cost[r, c]

    return groups, total_score

### 2.9 Evaluation (`evaluation.py`)

How good are the matches compared to the best possible?

- **`total_compatibility(groups, id_to_vec)`** — sum of `compatibility_score` over every (mentor, student) pair. Uses the same scoring function regardless of how `groups` was produced, so different strategies are directly comparable.
- **`ceiling_ratio(observed, ceiling)`** — fraction of the optimal (Hungarian) total that a result achieved; returns `0.0` instead of dividing by zero.

In [ ]:
def total_compatibility(groups: dict[str, list[str]], id_to_vec: dict) -> float:
    """Sum of cosine similarity across every (mentor, student) pair in
    `groups`. Comparable across different assignment strategies -- e.g. the
    `capacitated_assignment` ceiling vs. what a batch of live
    `recommend_for_student` calls actually produced -- since it's the same
    scoring function either way."""
    total = 0.0
    for alumnus_id, student_ids in groups.items():
        for sid in student_ids:
            total += compatibility_score(id_to_vec[alumnus_id], id_to_vec[sid])
    return total


def ceiling_ratio(observed_total: float, ceiling_total: float) -> float:
    """What fraction of the compatibility ceiling (e.g. from
    `capacitated_assignment` over the full live pool) a given result
    achieved. Returns 0.0 rather than raising on a zero ceiling (a
    degenerate/tiny input with no similarity at all) instead of letting a
    ZeroDivisionError surface."""
    if ceiling_total == 0:
        return 0.0
    return observed_total / ceiling_total

### 2.10 Behavioral match events (`events.py`)

`suggested` / `accepted` / `declined` events are mechanically loggable **actions** (not opinions), so they're safe to log directly.

- **`MatchEvent`** — the event schema a backend should implement: type, student id, alumnus id, a UTC ISO timestamp (auto-filled) and free-form metadata. `to_dict` / `from_dict` handle serialization.
- **`JsonlEventLog`** — minimal reference implementation: an append-only JSON-lines file. `read_all()` returns an empty list if the file doesn't exist yet. A production backend would likely use a database or event queue with the same shape.

In [ ]:
EventType = Literal["suggested", "accepted", "declined"]


@dataclass
class MatchEvent:
    event_type: EventType
    student_id: str
    alumnus_id: str
    timestamp: str = field(default_factory=lambda: datetime.now(timezone.utc).isoformat())
    metadata: dict[str, Any] = field(default_factory=dict)

    def to_dict(self) -> dict:
        return asdict(self)

    @classmethod
    def from_dict(cls, d: dict) -> "MatchEvent":
        return cls(**d)


class JsonlEventLog:
    """Append-only JSON-lines event log."""

    def __init__(self, path: str):
        self.path = path

    def append(self, event: MatchEvent) -> None:
        with open(self.path, "a") as f:
            f.write(json.dumps(event.to_dict()) + "\n")

    def read_all(self) -> list[MatchEvent]:
        try:
            with open(self.path) as f:
                return [MatchEvent.from_dict(json.loads(line)) for line in f if line.strip()]
        except FileNotFoundError:
            return []

### 2.11 Post-match feedback schema (`feedback_schema.py`)

Defines and **validates** the shape of a real human post-match survey response — it generates nothing and trains nothing. A future learned re-ranker ("Tier 2") is explicitly gated on real responses collected through a form built to this schema.

Fields and what they'd contribute to a future re-ranker:
- `satisfaction_rating` (1–5), `would_match_again` — explicit supervised labels.
- `sessions_held`, `relationship_status` — implicit engagement signal.
- `primary_goal_progress` — self-reported outcome tied to the student's stated goal.
- `free_text_comments` — qualitative review now, candidate feature source later.

`__post_init__` rejects invalid values (unknown role/status/progress, rating outside 1–5, negative sessions) with a `ValueError`.

In [ ]:
RespondentRole = Literal["student", "alumni"]
RelationshipStatus = Literal["ongoing", "ended", "never_started"]
GoalProgress = Literal["none", "some", "significant"]

_VALID_ROLES = {"student", "alumni"}
_VALID_STATUS = {"ongoing", "ended", "never_started"}
_VALID_PROGRESS = {"none", "some", "significant"}


@dataclass
class FeedbackForm:
    match_id: str
    respondent_role: RespondentRole

    # Explicit supervised-learning labels -- the clearest positive/negative
    # signal a re-ranker could eventually train on directly.
    satisfaction_rating: int  # 1 (poor) - 5 (excellent)
    would_match_again: bool

    # Implicit engagement signal -- often more reliable than a single
    # rating, and still present even if someone skips the rating question.
    sessions_held: int
    relationship_status: RelationshipStatus

    # Self-reported outcome, tying match quality back to the goal the
    # student stated at opt-in (synthesize_student_fields' career_goal_text
    # today; a real career-goals-form field later).
    primary_goal_progress: GoalProgress

    # Free text: not directly trainable without further NLP work, but
    # valuable for qualitative pilot review now and a candidate feature
    # source later (sentiment, topic extraction).
    free_text_comments: Optional[str] = None

    submitted_at: str = field(default_factory=lambda: datetime.now(timezone.utc).isoformat())

    def __post_init__(self):
        if self.respondent_role not in _VALID_ROLES:
            raise ValueError(f"respondent_role must be one of {_VALID_ROLES}, got {self.respondent_role!r}")
        if not (1 <= self.satisfaction_rating <= 5):
            raise ValueError(f"satisfaction_rating must be 1-5, got {self.satisfaction_rating!r}")
        if self.sessions_held < 0:
            raise ValueError(f"sessions_held must be >= 0, got {self.sessions_held!r}")
        if self.relationship_status not in _VALID_STATUS:
            raise ValueError(f"relationship_status must be one of {_VALID_STATUS}, got {self.relationship_status!r}")
        if self.primary_goal_progress not in _VALID_PROGRESS:
            raise ValueError(
                f"primary_goal_progress must be one of {_VALID_PROGRESS}, got {self.primary_goal_progress!r}"
            )

    def to_dict(self) -> dict:
        return asdict(self)

    @classmethod
    def from_dict(cls, d: dict) -> "FeedbackForm":
        return cls(**d)

### 2.12 Export for manual review (`export.py`)

**`export_matches_to_json(...)`** writes the final matches to a human-readable JSON file for sanity-checking (not for the app to consume):

- `run_stats` — summary numbers for the run.
- `unmatched_students` — profiles of anyone left without a mentor.
- `groups` — each mentor's profile, `group_size`, and matched students (with a rounded `compatibility_score` when `id_to_vec` is supplied).

Only the `REVIEW_FIELDS` present on the DataFrame are included, and NaN fields are skipped so the output is strict, standards-compliant JSON (a literal `NaN` would break e.g. JavaScript's `JSON.parse`).

In [ ]:
REVIEW_FIELDS = [
    "Name",
    "Major",
    "Hobbies",
    "Country",
    "State/Province",
    "Unique Quality",
    "target_track",
    "career_goal_text",
    "mentor_tracks",
    "capacity",
]


def export_matches_to_json(
    groups: dict[str, list[str]],
    df: pd.DataFrame,
    path: str,
    id_to_vec: dict | None = None,
    unmatched_ids: list[str] | None = None,
    run_stats: dict | None = None,
) -> str:
    """Writes final matches to `path`, with readable profile fields per
    person (whichever of REVIEW_FIELDS are present -- works whether or not
    the synthetic opt-in fields exist on `df`) and, when `id_to_vec` is
    supplied, each match's compatibility score for eyeballing quality.
    """
    df_indexed = df.set_index("person_id")
    present_fields = [c for c in REVIEW_FIELDS if c in df_indexed.columns]

    def profile(pid: str) -> dict:
        if pid not in df_indexed.index:
            return {"id": pid}
        row = df_indexed.loc[pid]
        # Skip NaN fields (e.g. a student-only column on an alumnus's row,
        # or vice versa) rather than including them -- `json.dump` would
        # otherwise emit a literal `NaN` token, which standard JSON doesn't
        # support and a strict parser (e.g. JS `JSON.parse`) would reject.
        # List-valued cells (Hobbies, mentor_tracks, requested_support) are
        # always "present" when not NaN -- checked first so `pd.notna` (which
        # returns an array, not a scalar, for list-like input) is never
        # called on one.
        return {
            "id": pid,
            **{c: row[c] for c in present_fields if isinstance(row[c], list) or pd.notna(row[c])},
        }

    match_list = []
    for alumnus_id, student_ids in groups.items():
        entry = profile(alumnus_id)
        students = []
        for sid in student_ids:
            s = profile(sid)
            if id_to_vec and alumnus_id in id_to_vec and sid in id_to_vec:
                s["compatibility_score"] = round(compatibility_score(id_to_vec[alumnus_id], id_to_vec[sid]), 4)
            students.append(s)
        entry["group_size"] = len(students)
        entry["matched_students"] = students
        match_list.append(entry)

    output = {
        "run_stats": run_stats or {},
        "unmatched_students": [profile(sid) for sid in (unmatched_ids or [])],
        "groups": match_list,
    }

    with open(path, "w") as f:
        json.dump(output, f, indent=2, default=str)
    return path

## 3. End-to-end demo (`run_demo.py`)

The cells below run the full pipeline against the real `student_profiles.json` population, one step per cell:

load → split alumni/students → sample opt-in → synthesize form fields → train/live split → fit features → **batch ceiling** → **live greedy simulation** → compare → filtered recommendation → export → event logging → feedback-form example.

### Step 1 — Load the full population
Loads all profiles and proxy-splits them into alumni-eligible (`Year == "Senior"`) and student pools.

In [ ]:
print("1. Loading full population...")
df = load_data(DATA_PATH)
alumni_pool, student_pool = split_alumni_students(df, alumni_years=("Senior",))
print(f"   {len(alumni_pool)} alumni-eligible (Year == Senior), {len(student_pool)} students, in the full population")

### Step 2 — Sample the opted-in pilot population
No real opt-in data exists yet, so a random sample of each pool stands in for the people who signed up.

In [ ]:
print("2. Sampling the opted-in pilot population (no real opt-in data exists yet -- random sample stands in)...")
opted_alumni, opted_students = sample_opt_in_population(
    alumni_pool, student_pool, seed=SEED, alumni_n=ALUMNI_OPT_IN_N, student_n=STUDENT_OPT_IN_N
)
print(f"   {len(opted_alumni)} opted-in alumni, {len(opted_students)} opted-in students")

### Step 3 — Synthesize opt-in-form fields
Adds the career-goal fields to students and the capacity/track/availability fields to alumni. **Synthetic, illustrative data** — see §2.4.

In [ ]:
print("3. Synthesizing opt-in-form fields (career goals, capacity, preferences)...")
print("   NOTE: synthetic/illustrative data standing in for a real form -- see synthetic_profile.py")
opted_alumni = synthesize_alumni_fields(opted_alumni, seed=SEED)
opted_students = synthesize_student_fields(opted_students, seed=SEED)

### Step 4 — Train / live split
Holds out `LIVE_FRAC` of each opted-in pool as "live" traffic, split independently for alumni and students.

In [ ]:
print("4. Splitting the opted-in population into train / live...")
train_alumni, live_alumni = train_live_split(opted_alumni, live_frac=LIVE_FRAC, seed=SEED)
train_students, live_students = train_live_split(opted_students, live_frac=LIVE_FRAC, seed=SEED)
print(f"   alumni: {len(train_alumni)} train / {len(live_alumni)} live")
print(f"   students: {len(train_students)} train / {len(live_students)} live")

### Step 5 — Fit the feature encoder and vectorize the live population
- Fits `FeatureEncoder` on the **train** split only (so live users are unseen), with location slightly down-weighted (0.75) and track slightly up-weighted (1.25).
- Transforms the live alumni and students into sparse vectors.
- Builds `live_id_to_vec` (person id → 1-row vector), the id lists, and the `capacity` lookup used by every later step.

In [ ]:
print("5. Fitting the feature encoder on the train split only...")
encoder = FeatureEncoder(Weights(major=1.0, hobbies=1.0, trait=1.0, location=0.75, track=1.25)).fit(
    pd.concat([train_alumni, train_students], ignore_index=True)
)
live_alumni_vecs = encoder.transform(live_alumni)
live_student_vecs = encoder.transform(live_students)

live_id_to_vec = {}
for i, pid in enumerate(live_alumni["person_id"]):
    live_id_to_vec[pid] = live_alumni_vecs[i]
for i, pid in enumerate(live_students["person_id"]):
    live_id_to_vec[pid] = live_student_vecs[i]

live_alumni_ids = live_alumni["person_id"].tolist()
live_student_ids = live_students["person_id"].tolist()
capacity = dict(zip(live_alumni["person_id"], live_alumni["capacity"]))

### Step 6 — Batch cold-start assignment (the compatibility ceiling)
Runs the Hungarian `capacitated_assignment` over the live population. Its total score is the best achievable under the capacity constraints.

In [ ]:
print("6. Batch cold-start assignment over the live population (the compatibility ceiling)...")
ceiling_groups, ceiling_total = capacitated_assignment(
    live_student_ids, live_student_vecs, live_alumni_ids, live_alumni_vecs, capacity
)
ceiling_matched = sum(len(v) for v in ceiling_groups.values())
print(f"   {ceiling_matched}/{len(live_students)} students matched, total compatibility = {ceiling_total:.2f}")

### Step 7 — Simulate the live product flow
Builds a `MentorIndex` once, then processes students **one at a time**, taking each student's single top recommendation and decrementing that mentor's remaining capacity — what `recommend_for_student` would produce in production. The result is scored with `total_compatibility` and compared to the ceiling via `ceiling_ratio`. The gap is the real cost of serving matches online instead of solving one big batch optimization.

In [ ]:
print("7. Simulating the actual live product flow: one student at a time, greedy top pick...")
print("   (this is what recommend_for_student would produce in production, not a batch sweep)")
live_index = MentorIndex(live_alumni_ids, live_alumni_vecs)
greedy_remaining_capacity = dict(capacity)
greedy_groups = {aid: [] for aid in live_alumni_ids}
for i, sid in enumerate(live_student_ids):
    picks = recommend_for_student(
        live_student_vecs[i], live_index, live_id_to_vec, greedy_remaining_capacity, show_k=1
    )
    if picks:
        aid = picks[0]
        greedy_groups[aid].append(sid)
        greedy_remaining_capacity[aid] -= 1

greedy_matched = sum(len(v) for v in greedy_groups.values())
greedy_total = total_compatibility(greedy_groups, live_id_to_vec)
print(f"   {greedy_matched}/{len(live_students)} students matched, total compatibility = {greedy_total:.2f}")
print(f"   greedy-live achieves {100 * ceiling_ratio(greedy_total, ceiling_total):.1f}% of the batch-optimal ceiling")

### Step 8 — Example recommendation with a hard filter
For the first live student, restricts candidates to mentors whose `mentor_tracks` include the student's `target_track`, then returns an MMR-diversified top 3.

In [ ]:
print("8. Example live recommendation with a hard filter...")
example_student_idx = 0
example_student_id = live_student_ids[example_student_idx]
example_track = live_students.iloc[example_student_idx]["target_track"]
mentor_df_indexed = live_alumni.set_index("person_id")
filtered_picks = recommend_for_student(
    live_student_vecs[example_student_idx],
    live_index,
    live_id_to_vec,
    dict(capacity),
    mentor_df_indexed=mentor_df_indexed,
    hard_filters={"mentor_tracks": example_track},
    show_k=3,
)
print(f"   student {example_student_id} (wants '{example_track}') -> recommended: {filtered_picks or '(no eligible mentors)'}")

### Step 9 — Export the ceiling assignment for manual review
Collects run statistics, finds students the ceiling assignment left unmatched, and writes everything to `OUTPUT_PATH` via `export_matches_to_json`.

In [ ]:
print("9. Exporting the ceiling assignment to JSON for manual review...")
run_stats = {
    "n_alumni_opted_in": len(opted_alumni),
    "n_students_opted_in": len(opted_students),
    "n_alumni_live": len(live_alumni),
    "n_students_live": len(live_students),
    "ceiling_students_matched": ceiling_matched,
    "ceiling_total_score": round(ceiling_total, 2),
    "greedy_live_students_matched": greedy_matched,
    "greedy_live_total_score": round(greedy_total, 2),
    "greedy_live_pct_of_ceiling": round(100 * ceiling_ratio(greedy_total, ceiling_total), 1),
}
unmatched = [sid for sid in live_student_ids if sid not in {s for ss in ceiling_groups.values() for s in ss}]
live_population = pd.concat([live_alumni, live_students], ignore_index=True)
out_path = export_matches_to_json(
    ceiling_groups, live_population, OUTPUT_PATH, id_to_vec=live_id_to_vec, unmatched_ids=unmatched, run_stats=run_stats,
)
print(f"   wrote {out_path}")

### Step 10 — Log example match events
Appends a `suggested` and an `accepted` event for the Step 8 student's top pick to `EVENT_LOG_PATH`, then reads the log back. Note: the log is append-only, so re-running this cell adds more events.

In [ ]:
print("10. Logging example match events (suggested/accepted -- mechanical actions, safe to log directly)...")
event_log = JsonlEventLog(EVENT_LOG_PATH)
if filtered_picks:
    top_pick = filtered_picks[0]
    event_log.append(MatchEvent(event_type="suggested", student_id=example_student_id, alumnus_id=top_pick))
    event_log.append(MatchEvent(event_type="accepted", student_id=example_student_id, alumnus_id=top_pick))
    print(f"   logged 2 events to {EVENT_LOG_PATH}: {[e.event_type for e in event_log.read_all()]}")

### Step 11 — Illustrative feedback form
Constructs one `FeedbackForm` to demonstrate schema validation. **Not real data and never used for training.**

In [ ]:
print("11. One illustrative FeedbackForm example (schema validation only -- NOT real data, NOT used for training)...")
example_feedback = FeedbackForm(
    match_id=f"{example_student_id}-{filtered_picks[0]}" if filtered_picks else "example",
    respondent_role="student",
    satisfaction_rating=5,
    would_match_again=True,
    sessions_held=2,
    relationship_status="ongoing",
    primary_goal_progress="some",
    free_text_comments="(illustrative example only, not a real response)",
)
print(f"   {example_feedback}")

## 4. Where to go from here

**What to tune**

| What | Where |
|---|---|
| Pilot size | `ALUMNI_OPT_IN_N`, `STUDENT_OPT_IN_N` (§1.3) |
| Train/live split | `LIVE_FRAC` |
| Feature weights | `Weights(...)` in Step 5 |
| Candidate breadth / shortlist size | `recommend_for_student(top_k_candidates=, show_k=)` |
| Diversity vs. compatibility | `recommend_for_student(lam=)` — 1.0 = pure compatibility, 0.0 = pure diversity |

**Next steps:** replace Steps 2–3 with real opt-in form submissions once those forms exist (nothing else needs to change), and once real `FeedbackForm` responses and `MatchEvent` logs accumulate, train a Tier 2 re-ranker (e.g. logistic regression) on top of the same feature vectors.